# Individual Task 1 - Part 1.3 Data Analysis
## Retail Commercial Performance: Operational and Customer-Level Insights

This notebook covers the data analysis part of Individual Task 1, tied to
the Commercial Data Analyst role at Coles Group I picked for Part 1.1.

The idea I went with is looking at retail commercial performance from two
different angles using two separate datasets, one at the store/operational
level and one at the individual customer level, and seeing whether the
same two models pick up similar or different patterns in each. I kept the
same two models across both datasets on purpose, mainly because the course
Q&A said if you're telling one story across two datasets you should
generally stick to the same models rather than switching them up.

**Datasets used:**
1. Walmart Store Sales Forecasting (Kaggle): weekly sales across 45
   stores, merged with store metadata and some regional/promotional
   features (421,570 rows after merging the three files together).
2. Online Retail II (UCI): real transaction data from a UK online
   retailer, which I rolled up into per-customer RFM features
   (1,067,371 raw transactions, roughly 5,900 customers after cleaning).

**Models:** Random Forest and a Neural Network (MLP), same two on both
datasets.


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    accuracy_score, confusion_matrix
)

RANDOM_STATE = 42
DATA_DIR = "../data"


## 2. Dataset 1: Walmart Store Sales Forecasting

### 2.1 Loading and merging

This one comes as 3 separate csv files that need joining together:
`train.csv` has the actual weekly sales, `stores.csv` has basic info
about each store (type and size), and `features.csv` has extra stuff
like temperature, fuel price, promo markdowns, CPI and unemployment for
each store/week combo.

One thing I ran into early on: `features.csv` has its own `IsHoliday`
column that's basically the same info as the one already in `train.csv`.
If I merge without dropping one of them first, pandas just renames both
to `IsHoliday_x` and `IsHoliday_y` which is confusing, so I drop the
duplicate from features before merging.


In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=["Date"])
features = pd.read_csv(f"{DATA_DIR}/features.csv", parse_dates=["Date"])
stores = pd.read_csv(f"{DATA_DIR}/stores.csv")

# drop the duplicate IsHoliday col from features before merging
features = features.drop(columns=["IsHoliday"])
walmart = train.merge(stores, on="Store", how="left")
walmart = walmart.merge(features, on=["Store", "Date"], how="left")

print(walmart.shape)
walmart.head()


(421570, 16)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106


### 2.2 Cleaning

Ran into a few things here worth explaining:

The `MarkDown1` to `MarkDown5` columns are empty for a huge chunk of the
data. At first I thought this was just missing data, but looking into it
more, Walmart only started actually tracking promotional markdowns from
around November 2011, so for earlier weeks it's not "unknown", it
genuinely means no promo was running. Filling these with the column mean
would be wrong since that implies "some promo happened but we don't know
how much" when really it should just be 0.

There were also a handful of missing `CPI` and `Unemployment` values near
the tail end of the date range (features.csv goes slightly further than
train.csv). These don't change much from week to week within the same
store, so I just forward-filled them per store rather than dropping those
rows.

Also dropped rows where `Weekly_Sales` is negative or zero, since these
look like returns or data corrections rather than genuine sales, and they
don't really fit the "high sales vs low sales" classification I'm trying
to build.


In [3]:
before = len(walmart)
walmart = walmart[walmart["Weekly_Sales"] > 0].copy()

# fill markdown NAs with 0 - see note above, NA here just means no promo yet
markdown_cols = [c for c in walmart.columns if c.startswith("MarkDown")]
walmart[markdown_cols] = walmart[markdown_cols].fillna(0)

# forward fill CPI/Unemployment per store since they barely move week to week
walmart = walmart.sort_values(["Store", "Date"])
walmart[["CPI", "Unemployment"]] = (
    walmart.groupby("Store")[["CPI", "Unemployment"]].ffill()
)
walmart = walmart.dropna(subset=["CPI", "Unemployment"])  # drop whatever's left with no value to ffill from

print(f"{before} rows -> {len(walmart)} rows after cleaning ({before - len(walmart)} dropped)")


421570 rows -> 420212 rows after cleaning (1358 dropped)


### 2.3 Feature engineering and building the target column

For the classification target I went with `HighSales`: basically, was
this store/department/week in the top 25% of sales compared to other
weeks *in that same department*. I deliberately did this per-department
rather than across the whole dataset, because departments vary a lot in
how much they sell overall (think groceries vs. seasonal decor). If I'd
used one flat cutoff across everything, the model would mostly just be
learning "which departments are big" instead of "which weeks were
actually strong performers".


In [4]:
walmart["Month"] = walmart["Date"].dt.month
walmart["WeekOfYear"] = walmart["Date"].dt.isocalendar().week.astype(int)
walmart["TotalMarkDown"] = walmart[markdown_cols].sum(axis=1)
walmart["HasPromotion"] = (walmart["TotalMarkDown"] > 0).astype(int)

# top 25% of sales WITHIN each department, not across the whole dataset
dept_q75 = walmart.groupby("Dept")["Weekly_Sales"].transform(lambda x: x.quantile(0.75))
walmart["HighSales"] = (walmart["Weekly_Sales"] >= dept_q75).astype(int)

walmart["HighSales"].value_counts(normalize=True).round(3)


HighSales
0    0.75
1    0.25
Name: proportion, dtype: float64

## 3. Dataset 2: Online Retail II

### 3.1 Loading

This one comes as an Excel file with two sheets, one for each year of
transactions. Just stacking them into one table with `pd.concat`.


In [5]:
xl = pd.ExcelFile(f"{DATA_DIR}/online_retail_II.xlsx")
retail_raw = pd.concat(
    [pd.read_excel(xl, sheet_name=s) for s in xl.sheet_names],
    ignore_index=True
)
print(retail_raw.shape)
retail_raw.head()


(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### 3.2 Cleaning

A few things to filter out here: invoices starting with `C` are
cancellations, not real purchases, so those get removed. Rows missing a
`Customer ID` can't really be used since I'm building customer-level
features later and there's no way to attach an anonymous transaction to
a customer. Also dropped rows with 0 or negative quantity/price since
those look like data entry mistakes or adjustment entries rather than
actual sales.


In [6]:
before = len(retail_raw)
retail = retail_raw[~retail_raw["Invoice"].astype(str).str.startswith("C")]
retail = retail.dropna(subset=["Customer ID"])
retail = retail[(retail["Quantity"] > 0) & (retail["Price"] > 0)]
retail["Customer ID"] = retail["Customer ID"].astype(int)
retail["LineTotal"] = retail["Quantity"] * retail["Price"]

print(f"{before} rows -> {len(retail)} rows after cleaning ({before - len(retail)} dropped)")


1067371 rows -> 805549 rows after cleaning (261822 dropped)


### 3.3 Building RFM features and the target column

Rolling the transaction-level data up to customer level using RFM,
Recency, Frequency, Monetary. This is a pretty standard approach for
customer segmentation (the paper that originally published this dataset,
Chen, Sain & Guo, 2012, actually uses this exact method).

For the target, `HighValue` flags customers sitting in the top 25% by how
much they've spent in total.

One thing I had to be careful about: `Monetary` is literally what I use
to *build* the `HighValue` label, so it can't also be used as a model
input. If it were, the model would just be reading the answer straight
off the label instead of learning anything, which would make the results
look artificially perfect for no real reason (this is called target
leakage, caught it while setting the features up below).


In [7]:
snapshot_date = retail["InvoiceDate"].max() + pd.Timedelta(days=1)

# RFM approach follows Chen, Sain & Guo (2012), who published this dataset
rfm = retail.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("LineTotal", "sum"),
    DistinctProducts=("StockCode", "nunique"),
    Country=("Country", "first"),
).reset_index()

# top 25% by spend = high value customer (same logic as the walmart target)
monetary_q75 = rfm["Monetary"].quantile(0.75)
rfm["HighValue"] = (rfm["Monetary"] >= monetary_q75).astype(int)
rfm["IsUK"] = (rfm["Country"] == "United Kingdom").astype(int)  # most customers are UK-based anyway

print(rfm.shape)
rfm["HighValue"].value_counts(normalize=True).round(3)


(5878, 8)


HighValue
0    0.75
1    0.25
Name: proportion, dtype: float64

## 4. Modelling

Both targets end up with roughly a 75/25 split since they're both built
off a top-25% threshold. This matters because a model that just guesses
"not high" every single time would already be right 75% of the time
without actually learning anything useful. So accuracy on its own isn't
a great metric to lean on here, I'm reporting precision, recall, F1 and
ROC-AUC instead since those actually show whether the model is picking
up a real signal or just exploiting the class imbalance.

Same two models on both datasets: Random Forest and a Neural Network
(MLP), so the results can be compared properly across the two datasets.


In [8]:
# just a small helper so I'm not copy pasting the same metric calculations
# twice for every model/dataset combo
def evaluate(y_true, y_pred, y_proba, model_name, dataset_name):
    metrics = {
        "dataset": dataset_name,
        "model": model_name,
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "precision": round(precision_score(y_true, y_pred), 4),
        "recall": round(recall_score(y_true, y_pred), 4),
        "f1": round(f1_score(y_true, y_pred), 4),
        "roc_auc": round(roc_auc_score(y_true, y_proba), 4),
    }
    print(f"--- {dataset_name} | {model_name} ---")
    for k, v in metrics.items():
        if k not in ("dataset", "model"):
            print(f"  {k:>10}: {v}")
    print(f"  confusion matrix:\n{confusion_matrix(y_true, y_pred)}\n")
    return metrics


def train_evaluate(X, y, feature_cols, dataset_name):
    # stratify so the train/test split keeps roughly the same 75/25 balance
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    results = []

    # class_weight=balanced because of the 75/25 split - without this the
    # forest just leans toward predicting the majority class more than it should
    # using random forest here - see Breiman (2001) for the original method
    rf = RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=20,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    rf_pred, rf_proba = rf.predict(X_test), rf.predict_proba(X_test)[:, 1]
    results.append(evaluate(y_test, rf_pred, rf_proba, "Random Forest", dataset_name))

    # quick look at which features the forest is actually relying on
    importances = pd.Series(rf.feature_importances_, index=feature_cols)\
        .sort_values(ascending=False)
    print(f"Feature importances ({dataset_name}):")
    print(importances.round(3).to_string(), "\n")

    # early_stopping=True so it stops training once validation loss stops improving,
    # otherwise it just runs the full 500 iterations regardless
    # neural net trained with backprop - see Rumelhart, Hinton & Williams (1986)
    nn = MLPClassifier(
        hidden_layer_sizes=(64, 32), activation="relu", max_iter=500,
        early_stopping=True, random_state=RANDOM_STATE
    )
    nn.fit(X_train_scaled, y_train)
    nn_pred = nn.predict(X_test_scaled)
    nn_proba = nn.predict_proba(X_test_scaled)[:, 1]
    results.append(evaluate(y_test, nn_pred, nn_proba, "Neural Network", dataset_name))

    return results, importances


### 4.1 Walmart model

Not using `Store` or `Dept` as raw features here, since the target is
already relative to each department (top 25% within that dept), the
department ID on its own wouldn't add much info, and store ID is more of
an identifier than something with real predictive meaning by itself.
`Type` gets one-hot encoded since it's categorical (A/B/C), everything
else is either already numeric or a count I built earlier.


In [9]:
walmart_dummies = pd.get_dummies(walmart, columns=["Type"], drop_first=True)

walmart_features = [
    "Size", "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "TotalMarkDown", "HasPromotion", "Month", "WeekOfYear", "IsHoliday",
    "Type_B", "Type_C",
]
X_walmart = walmart_dummies[walmart_features].copy()
X_walmart["IsHoliday"] = X_walmart["IsHoliday"].astype(int)  # was True/False, needs to be numeric
y_walmart = walmart_dummies["HighSales"]

walmart_results, walmart_importance = train_evaluate(
    X_walmart, y_walmart, walmart_features, "Walmart (store sales)"
)


--- Walmart (store sales) | Random Forest ---
    accuracy: 0.7579
   precision: 0.5101
      recall: 0.8208
          f1: 0.6292
     roc_auc: 0.8583
  confusion matrix:
[[58042 20725]
 [ 4710 21576]]

Feature importances (Walmart (store sales)):
Size             0.482
CPI              0.143
Unemployment     0.126
Type_C           0.062
Type_B           0.056
WeekOfYear       0.031
Temperature      0.030
Fuel_Price       0.028
TotalMarkDown    0.021
Month            0.016
HasPromotion     0.004
IsHoliday        0.002 

--- Walmart (store sales) | Neural Network ---
    accuracy: 0.8142
   precision: 0.669
      recall: 0.5093
          f1: 0.5783
     roc_auc: 0.8523
  confusion matrix:
[[72143  6624]
 [12898 13388]]



--- Walmart (store sales) | Neural Network ---
    accuracy: 0.8142
   precision: 0.669
      recall: 0.5093
          f1: 0.5783
     roc_auc: 0.8523
  confusion matrix:
[[72143  6624]
 [12898 13388]]



### 4.2 Online Retail II model

Same idea here, just a smaller feature set since RFM only really gives
me 3 numeric features plus the country flag (and again, Monetary is left
out on purpose - see the note above about target leakage).


In [10]:
retail_features = ["Recency", "Frequency", "DistinctProducts", "IsUK"]
X_retail = rfm[retail_features].copy()
y_retail = rfm["HighValue"]

retail_results, retail_importance = train_evaluate(
    X_retail, y_retail, retail_features, "Online Retail II (customers)"
)


--- Online Retail II (customers) | Random Forest ---
    accuracy: 0.9007
   precision: 0.75
      recall: 0.9049
          f1: 0.8202
     roc_auc: 0.9602
  confusion matrix:
[[991 111]
 [ 35 333]]

Feature importances (Online Retail II (customers)):
Frequency           0.598
DistinctProducts    0.312
Recency             0.078
IsUK                0.013 

--- Online Retail II (customers) | Neural Network ---
    accuracy: 0.9068
   precision: 0.8787
      recall: 0.7283
          f1: 0.7964
     roc_auc: 0.9511
  confusion matrix:
[[1065   37]
 [ 100  268]]



--- Online Retail II (customers) | Neural Network ---
    accuracy: 0.9068
   precision: 0.8787
      recall: 0.7283
          f1: 0.7964
     roc_auc: 0.9511
  confusion matrix:
[[1065   37]
 [ 100  268]]



## 5. Putting both sets of results together

In [11]:
all_results = walmart_results + retail_results
results_df = pd.DataFrame(all_results)
results_df


,dataset,model,accuracy,precision,recall,f1,roc_auc
0,Walmart (store sales),Random Forest,0.7579,0.5101,0.8208,0.6292,0.8583
1,Walmart (store sales),Neural Network,0.8142,0.6690,0.5093,0.5783,0.8523
2,Online Retail II (customers),Random Forest,0.9007,0.7500,0.9049,0.8202,0.9602
3,Online Retail II (customers),Neural Network,0.9068,0.8787,0.7283,0.7964,0.9511


## 6. What I'm taking away from this

**Store size and macro conditions matter way more than promotions, and
the same pattern shows up on the customer side too.** Looking at the
Random Forest feature importances, store `Size` alone is doing about half
the work in the Walmart model, with CPI and Unemployment together adding
another roughly 27%. Meanwhile promo markdown activity barely registers, under
3% combined. Over on the Online Retail side, it's `Frequency` (roughly 60%) and
`DistinctProducts` (roughly 31%) driving most of the prediction, with `Recency`
only around 8%. What's interesting is these are two completely separate
datasets from different countries and different types of retail, and
they're both pointing at the same underlying idea: steady, structural
stuff (how big a store is, how often a customer actually buys) predicts
performance a lot better than short-term levers like a one-off promotion
or how recently someone last shopped. If I were actually presenting this
to a commercial team, I'd probably say reporting should lean more on
these structural signals rather than obsessing over promo-level metrics.

**Random Forest and the Neural Network behave consistently across both
datasets, just not in the way I expected.** In both cases Random Forest
comes out with noticeably higher recall than the Neural Network, and the
Neural Network comes out ahead on precision instead. ROC-AUC ends up
pretty close between the two models on each dataset, so overall they're
similarly good at telling the classes apart, it's really about where
each one draws the line for deciding "yes this is high" vs "no it's not".
Since this same pattern shows up on both datasets and they're otherwise
nothing alike, it seems more like something to do with how these two
algorithms behave under class imbalance in general rather than anything
specific to walmart or retail data. Practically speaking, if missing an
actual high performer is worse than incorrectly flagging one (which feels
true for something like inventory planning), Random Forest looks like the
better pick of the two given its recall advantage.

**Quick note on why I didn't just use accuracy.** Both targets are built
off a 75/25 split (top 25% threshold), so a model that always predicts
"not high" would score 75% accuracy without learning anything real. That's
basically why precision, recall, F1 and ROC-AUC are the numbers that
actually matter here, not accuracy by itself.
